### Data ingestion -> VectorDB

In [17]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [18]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: be_information-technology_second-year-se-semester-34-nep-2020 (1).pdf
  ✓ Loaded 70 pages

Processing: C11_2203084_Prateek Khemchandani.pdf
  ✓ Loaded 59 pages

Processing: Prateek Khemchandani Resume-3.pdf
  ✓ Loaded 1 pages

Total documents loaded: 130


In [19]:
all_pdf_documents

[Document(metadata={'producer': '3.0.33 (5.1.21)', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2026-05-08T16:13:31+02:00', 'source': '..\\data\\pdf\\be_information-technology_second-year-se-semester-34-nep-2020 (1).pdf', 'total_pages': 70, 'page': 0, 'page_label': '1', 'source_file': 'be_information-technology_second-year-se-semester-34-nep-2020 (1).pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': '3.0.33 (5.1.21)', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2026-05-08T16:13:31+02:00', 'source': '..\\data\\pdf\\be_information-technology_second-year-se-semester-34-nep-2020 (1).pdf', 'total_pages': 70, 'page': 1, 'page_label': '2', 'source_file': 'be_information-technology_second-year-se-semester-34-nep-2020 (1).pdf', 'file_type': 'pdf'}, page_content='Copy forwarded for information and necessary action to :- \n \n1 The Deputy Registrar, (Admissions, Enrolment, Eligibility and Migration Dept)(AEM), \ndr@eligi.mu.ac.in \n \n2 The Deputy Registr

In [20]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [21]:
chunks=split_documents(all_pdf_documents)
chunks

Split 130 documents into 197 chunks

Example chunk:
Content: Copy forwarded for information and necessary action to :- 
 
1 The Deputy Registrar, (Admissions, Enrolment, Eligibility and Migration Dept)(AEM), 
dr@eligi.mu.ac.in 
 
2 The Deputy Registrar, Result ...
Metadata: {'producer': '3.0.33 (5.1.21)', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2026-05-08T16:13:31+02:00', 'source': '..\\data\\pdf\\be_information-technology_second-year-se-semester-34-nep-2020 (1).pdf', 'total_pages': 70, 'page': 1, 'page_label': '2', 'source_file': 'be_information-technology_second-year-se-semester-34-nep-2020 (1).pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': '3.0.33 (5.1.21)', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2026-05-08T16:13:31+02:00', 'source': '..\\data\\pdf\\be_information-technology_second-year-se-semester-34-nep-2020 (1).pdf', 'total_pages': 70, 'page': 1, 'page_label': '2', 'source_file': 'be_information-technology_second-year-se-semester-34-nep-2020 (1).pdf', 'file_type': 'pdf'}, page_content='Copy forwarded for information and necessary action to :- \n \n1 The Deputy Registrar, (Admissions, Enrolment, Eligibility and Migration Dept)(AEM), \ndr@eligi.mu.ac.in \n \n2 The Deputy Registrar, Result unit, Vidyanagari \ndrresults@exam.mu.ac.in \n \n3 The Deputy Registrar, Marks and Certificate Unit,. Vidyanagari \ndr.verification@mu.ac.in \n \n4 The Deputy Registrar, Appointment Unit, Vidyanagari \ndr.appointment@exam.mu.ac.in \n \n5 The Deputy Registrar, CAP Unit, Vidyanagari \ncap.exam@mu.ac.in \n \n6 The Deputy Registrar, College Affiliations & Development Department (CAD), \ndeputyre

### Embedding, VectorDB

In [22]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [23]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2443.50it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\PRATEEKK\AppData\Local\Temp\ipykernel_61596\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [24]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 36


In [25]:
chunks

[Document(metadata={'producer': '3.0.33 (5.1.21)', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2026-05-08T16:13:31+02:00', 'source': '..\\data\\pdf\\be_information-technology_second-year-se-semester-34-nep-2020 (1).pdf', 'total_pages': 70, 'page': 1, 'page_label': '2', 'source_file': 'be_information-technology_second-year-se-semester-34-nep-2020 (1).pdf', 'file_type': 'pdf'}, page_content='Copy forwarded for information and necessary action to :- \n \n1 The Deputy Registrar, (Admissions, Enrolment, Eligibility and Migration Dept)(AEM), \ndr@eligi.mu.ac.in \n \n2 The Deputy Registrar, Result unit, Vidyanagari \ndrresults@exam.mu.ac.in \n \n3 The Deputy Registrar, Marks and Certificate Unit,. Vidyanagari \ndr.verification@mu.ac.in \n \n4 The Deputy Registrar, Appointment Unit, Vidyanagari \ndr.appointment@exam.mu.ac.in \n \n5 The Deputy Registrar, CAP Unit, Vidyanagari \ncap.exam@mu.ac.in \n \n6 The Deputy Registrar, College Affiliations & Development Department (CAD), \ndeputyre

In [26]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings
embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 197 texts...


Batches: 100%|██████████| 7/7 [00:08<00:00,  1.27s/it]


Generated embeddings with shape: (197, 384)
Adding 197 documents to vector store...
Successfully added 197 documents to vector store
Total documents in collection: 233


### Retrieval pipeline (VectorDB)

In [27]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [28]:
rag_retriever

In [30]:
rag_retriever.retrieve("Syllabus of IT")

Retrieving documents for query: 'Syllabus of IT'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.15it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_5c5329bf_34',
  'content': 'should cover the maximum contents of the syllabus \n\uf0b7 Remaining questions will be mixed in nature (part (a) and part (b) of each question must be from different \nmodules. For example, if Q.2 has part (a) from Module 3 then part (b) must b e from any other Module \nrandomly selected from all the modules) \n\uf0b7 A total of four questions need to be answered.',
  'metadata': {'page_label': '19',
   'doc_index': 34,
   'page': 18,
   'total_pages': 70,
   'source': '..\\data\\pdf\\be_information-technology_second-year-se-semester-34-nep-2020 (1).pdf',
   'moddate': '2026-05-08T16:13:31+02:00',
   'content_length': 352,
   'source_file': 'be_information-technology_second-year-se-semester-34-nep-2020 (1).pdf',
   'creationdate': '',
   'file_type': 'pdf',
   'producer': '3.0.33 (5.1.21)',
   'creator': 'PyPDF'},
  'similarity_score': 0.05029785633087158,
  'distance': 0.9497021436691284,
  'rank': 1}]

In [37]:
rag_retriever.retrieve("End Semester Theory Examination")

Retrieving documents for query: 'End Semester Theory Examination'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 11.18it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_656e820f_33',
  'content': 'Applied Mathematics. This project should be graded for 10 marks depending on the performance of \nthe students. \n \nThe distribution of Term Work marks will be as follows– \n \n \n1. Attendance (Theory and Tutorial) 05marks \n2. Class Tutorials on entire syllabus 10marks \n3. Mini project 10marks \nAssessment: \nInternal Assessment Test (IAT) for 20 marks each: \n\uf0b7 IA will consist of Two Compulsory Internal Assessment Tests. Approximately 40% to 50% of the syllabus \ncontent must be covered in the IAT-I and the remaining 40% to 50% of the syllabus content must be covered in \nthe IAT-II. \nEnd Semester Theory Examination: \n\uf0d8 Question paper format \n\uf0b7 Question Paper will comprise a total of six questions each carrying 15 marks Q.1 will be compulsory and \nshould cover the maximum contents of the syllabus \n\uf0b7 Remaining questions will be mixed in nature (part (a) and part (b) of each question must be from different',
  'metada